# Laboratório — KDD, Machine Learning e Redes Neurais

Este notebook acompanha a unidade de **Big Data Analytics** e foi pensado para uso em sala de aula.

Vamos trabalhar com:

- **KDD** na prática;
- preparação de dados;
- prevenção de **data leakage**;
- classificação e regressão;
- modelos baseline;
- métricas de avaliação;
- TensorFlow / Keras;
- `MLPClassifier` do scikit-learn;
- comparação entre modelos;
- desafios progressivos;
- atividade final aberta.

## Datasets públicos utilizados

1. **Adult / Census Income — UCI Machine Learning Repository**  
   Usado para KDD, dados categóricos, dados ausentes e classificação.  
   **Na aula, o notebook procura primeiro o arquivo local `dados/adult.csv`.**

2. **Breast Cancer Wisconsin — scikit-learn**  
   Já vem disponível com o scikit-learn e não precisa ser baixado separadamente.

3. **California Housing**  
   Usado para regressão.  
   **Na aula, o notebook procura primeiro o arquivo local `dados/california_housing.csv`.**

> A regra deste laboratório é: **o fluxo principal não depende da internet**.  
> Há uma etapa opcional para preparar os arquivos locais antes da aula.

## Como usar este notebook

Há três tipos de atividade:

**GUIADO** — código pronto.

**DESAFIO** — código incompleto com `TODO`.

**GABARITO** — soluções passadas em outro arquivo.

Tente resolver os desafios antes de consultar o gabarito.

## 0. Instalação

Em um ambiente novo, instale as bibliotecas abaixo.

> O ideal é preparar o ambiente e os arquivos de dados com antecedência.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn tensorflow ucimlrepo

^C
Note: you may need to restart the kernel to use updated packages.


## 1. Importações

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    roc_auc_score,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

PASTA_DADOS = Path("dados")
ADULT_CSV = PASTA_DADOS / "adult.csv"
CALIFORNIA_CSV = PASTA_DADOS / "california_housing.csv"

## 1.1 Preparação dos datasets para a aula

O laboratório utiliza **arquivos locais** sempre que um dataset normalmente precisaria ser baixado.

Estrutura esperada:

```text
laboratorio_kdd_ml_deep_learning.ipynb
dados/
├── adult.csv
└── california_housing.csv
```

Isso evita que a aula dependa do OpenML, da UCI, do scikit-learn.

A célula abaixo apenas verifica se os arquivos já estão disponíveis.

In [2]:
print("Adult:")
print("  OK" if ADULT_CSV.exists() else f"  AUSENTE -> {ADULT_CSV}")

print("\nCalifornia Housing:")
print(
    "  OK"
    if CALIFORNIA_CSV.exists()
    else f"  AUSENTE -> {CALIFORNIA_CSV}"
)

Adult:
  AUSENTE -> dados\adult.csv

California Housing:
  AUSENTE -> dados\california_housing.csv


### Preparação opcional — executar antes da aula

A próxima célula **não é necessária durante a aula**.

Use-a apenas uma vez, em uma máquina com acesso à internet, para gerar os dois CSVs locais.

Por segurança, `BAIXAR_DADOS` começa como `False`.  
Altere para `True` somente quando quiser preparar os arquivos.

Depois, distribua a pasta `dados/` junto com o notebook.

In [ ]:
BAIXAR_DADOS = False

if BAIXAR_DADOS:
    PASTA_DADOS.mkdir(
        parents=True,
        exist_ok=True
    )

    try:
        from ucimlrepo import fetch_ucirepo

        adult_uci = fetch_ucirepo(id=2)

        df_adult_download = pd.concat(
            [
                adult_uci.data.features.copy(),
                adult_uci.data.targets.copy()
            ],
            axis=1
        )

        nome_alvo = adult_uci.data.targets.columns[0]

        df_adult_download = df_adult_download.rename(
            columns={
                nome_alvo: "class"
            }
        )

        df_adult_download.to_csv(
            ADULT_CSV,
            index=False
        )

        print(
            "Adult salvo em:",
            ADULT_CSV.resolve()
        )

    except Exception as erro:
        print(
            "Não foi possível preparar o Adult."
        )
        print(
            "Erro:",
            erro
        )

    try:
        from sklearn.datasets import fetch_california_housing

        california = fetch_california_housing(
            as_frame=True
        )

        california.frame.to_csv(
            CALIFORNIA_CSV,
            index=False
        )

        print(
            "California Housing salvo em:",
            CALIFORNIA_CSV.resolve()
        )

    except Exception as erro:
        print(
            "Não foi possível preparar o California Housing."
        )
        print(
            "Erro:",
            erro
        )
else:
    print(
        "Modo offline ativo. "
        "Nenhum download será realizado."
    )

# Parte 1 — KDD na prática com Adult / Census Income

Fluxo: **seleção → pré-processamento → transformação → mineração → avaliação**.

## 2. Seleção dos dados

Nesta etapa utilizamos o **Adult / Census Income**, originalmente disponibilizado pela UCI.

Para tornar a aula reprodutível, o conjunto é carregado do arquivo local:

```text
dados/adult.csv
```

A origem pública continua documentada, mas o laboratório não precisa consultar a internet.

In [ ]:
if not ADULT_CSV.exists():
    raise FileNotFoundError(
        "Arquivo 'dados/adult.csv' não encontrado. "
        "Prepare o dataset antes da aula usando a célula opcional "
        "de preparação ou distribua a pasta 'dados/' junto com o notebook."
    )

df_adult = pd.read_csv(
    ADULT_CSV
)

# Padroniza o nome da variável alvo caso o CSV tenha vindo
# de outra fonte com um nome diferente.
if "class" not in df_adult.columns:
    candidatos = [
        "income",
        "Income",
        "target",
        "Target"
    ]

    for candidato in candidatos:
        if candidato in df_adult.columns:
            df_adult = df_adult.rename(
                columns={
                    candidato: "class"
                }
            )
            break

if "class" not in df_adult.columns:
    raise ValueError(
        "A variável alvo do Adult não foi encontrada. "
        "O arquivo deve possuir a coluna 'class'."
    )

print(
    "Dataset carregado localmente:",
    ADULT_CSV.resolve()
)

print(
    "Dimensões:",
    df_adult.shape
)

display(
    df_adult.head()
)

## 3. Compreendendo o problema

In [ ]:
print('Colunas:')
print(df_adult.columns.tolist())
print('\nDistribuição da classe:')
print(df_adult['class'].value_counts(dropna=False))

## 4. Inspeção inicial

In [ ]:
df_adult.info()

In [ ]:
display(df_adult.describe(include='all').T)

## 5. Valores ausentes

Antes de remover, identifique onde estão e qual o impacto.

In [ ]:
nulos = df_adult.isna().sum().sort_values(ascending=False)
display(nulos[nulos > 0])

## DESAFIO 1 — Inspeção

1. Quais são as três colunas com mais valores ausentes?
2. Qual a proporção percentual em cada uma?
3. A variável alvo possui valores ausentes?

In [ ]:
# TODO: resolva aqui.

## 6. Separação entre atributos e alvo

A divisão acontece antes do ajuste dos transformadores para evitar leakage.

In [ ]:
X = df_adult.drop(columns='class')
y = df_adult['class'].astype(str).str.contains('>50K').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print('Treino:', X_train.shape)
print('Teste :', X_test.shape)

## 7. Colunas numéricas e categóricas

In [ ]:
colunas_numericas = X_train.select_dtypes(include='number').columns.tolist()
colunas_categoricas = X_train.select_dtypes(exclude='number').columns.tolist()
print('Numéricas:', colunas_numericas)
print('\nCategóricas:', colunas_categoricas)

## 8. Pipeline de pré-processamento

In [ ]:
pipeline_numerico = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

pipeline_categorico = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessador = ColumnTransformer([
    ('numerico', pipeline_numerico, colunas_numericas),
    ('categorico', pipeline_categorico, colunas_categoricas),
])

## 9. Baseline — Regressão Logística

In [ ]:
baseline = Pipeline([
    ('preprocessador', preprocessador),
    ('modelo', LogisticRegression(max_iter=1000)),
])

baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)
prob_baseline = baseline.predict_proba(X_test)[:, 1]

## 10. Avaliação do baseline

In [ ]:
print(classification_report(y_test, pred_baseline))
print('ROC AUC:', roc_auc_score(y_test, prob_baseline))

## 11. Matriz de confusão

In [ ]:
matriz = confusion_matrix(y_test, pred_baseline)
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de confusão — baseline')
plt.show()

## DESAFIO 2 — Interpretação

1. Quantos falsos positivos ocorreram?
2. Quantos falsos negativos ocorreram?
3. Em um problema real de concessão de crédito, qual erro poderia ser mais caro?
4. Acurácia sozinha seria suficiente?

In [ ]:
# TODO: escreva suas respostas como comentários.

## 12. Segundo modelo — Random Forest

In [ ]:
random_forest = Pipeline([
    ('preprocessador', preprocessador),
    ('modelo', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
])

random_forest.fit(X_train, y_train)
pred_rf = random_forest.predict(X_test)
prob_rf = random_forest.predict_proba(X_test)[:, 1]

## 13. Comparando modelos

In [ ]:
comparacao_adult = pd.DataFrame({
    'modelo': ['Logistic Regression', 'Random Forest'],
    'accuracy': [accuracy_score(y_test, pred_baseline), accuracy_score(y_test, pred_rf)],
    'precision': [precision_score(y_test, pred_baseline), precision_score(y_test, pred_rf)],
    'recall': [recall_score(y_test, pred_baseline), recall_score(y_test, pred_rf)],
    'roc_auc': [roc_auc_score(y_test, prob_baseline), roc_auc_score(y_test, prob_rf)],
})
display(comparacao_adult)

## DESAFIO 3 — Qual modelo escolher?

Analise métricas, custo computacional e interpretabilidade. Justifique.

In [ ]:
# TODO: escreva sua análise.

# Parte 2 — Classificação com Breast Cancer Wisconsin

Vamos comparar regressão logística, Random Forest, MLP e TensorFlow/Keras.

> **Observação:** o Breast Cancer Wisconsin é diferente dos dois datasets anteriores:  
> ele já acompanha o scikit-learn, portanto `load_breast_cancer()` não depende de download externo.

## 14. Carregando o dataset

In [ ]:
cancer = load_breast_cancer(as_frame=True)
df_cancer = cancer.frame.copy()
print('Dimensões:', df_cancer.shape)
display(df_cancer.head())

## 15. Separando treino e teste

In [ ]:
X_cancer = df_cancer.drop(columns='target')
y_cancer = df_cancer['target']
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cancer, y_cancer, test_size=0.20, random_state=RANDOM_STATE, stratify=y_cancer
)

## 16. Baseline com Logistic Regression

In [ ]:
cancer_logistica = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', LogisticRegression(max_iter=1000)),
])
cancer_logistica.fit(Xc_train, yc_train)
pred_log = cancer_logistica.predict(Xc_test)
print(classification_report(yc_test, pred_log))

## DESAFIO 4 — Random Forest

Crie um `RandomForestClassifier` com 200 árvores e compare accuracy, precision e recall.

In [ ]:
# TODO: implemente o Random Forest.

# Parte 3 — Redes neurais com scikit-learn

O `MLPClassifier` é útil para redes multicamadas simples, mas não substitui frameworks especializados de deep learning.

## 17. MLPClassifier

In [ ]:
mlp = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', MLPClassifier(
        hidden_layer_sizes=(128, 64), activation='relu', early_stopping=True,
        max_iter=300, random_state=RANDOM_STATE
    )),
])
mlp.fit(Xc_train, yc_train)
pred_mlp = mlp.predict(Xc_test)
print(classification_report(yc_test, pred_mlp))

## 18. Por que usamos StandardScaler?

In [ ]:
display(X_cancer.describe().T[['mean', 'std', 'min', 'max']].head(10))

## DESAFIO 5 — Retire o scaler

Treine outro `MLPClassifier` sem `StandardScaler` e compare convergência e métricas.

In [ ]:
# TODO: crie mlp_sem_scaler e compare.

# Parte 4 — Rede Neural com TensorFlow / Keras

## 19. Importando TensorFlow

In [ ]:
import tensorflow as tf
tf.random.set_seed(RANDOM_STATE)
print('TensorFlow:', tf.__version__)

## 20. Preparação dos dados

In [ ]:
scaler_tf = StandardScaler()
Xc_train_scaled = scaler_tf.fit_transform(Xc_train)
Xc_test_scaled = scaler_tf.transform(Xc_test)
print(Xc_train_scaled.shape)

## 21. Construindo a rede

In [ ]:
modelo_tf = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(Xc_train_scaled.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
modelo_tf.summary()

## 22. Compilação

In [ ]:
modelo_tf.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ],
)

## 23. Early Stopping

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

## 24. Treinamento

In [ ]:
historico = modelo_tf.fit(
    Xc_train_scaled, yc_train,
    validation_split=0.20,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=0,
)
print('Épocas executadas:', len(historico.history['loss']))

## 25. Curva de treinamento

In [ ]:
historico_df = pd.DataFrame(historico.history)
historico_df[['loss', 'val_loss']].plot(figsize=(9, 5))
plt.title('Loss — treino x validação')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.show()

## 26. Avaliação

In [ ]:
resultado_tf = modelo_tf.evaluate(Xc_test_scaled, yc_test, verbose=0, return_dict=True)
display(pd.Series(resultado_tf, name='resultado'))

## 27. Probabilidade → classe

In [ ]:
prob_tf = modelo_tf.predict(Xc_test_scaled, verbose=0).ravel()
pred_tf = (prob_tf >= 0.50).astype(int)
print(classification_report(yc_test, pred_tf))

## DESAFIO 6 — Threshold

Teste 0.30, 0.50 e 0.70. Compare precision e recall.

In [ ]:
# TODO: compare os thresholds.

## DESAFIO 7 — Rede maior é sempre melhor?

Teste uma rede 128 → 64 → 32 com Dropout 0.30 e compare.

In [ ]:
# TODO: construa modelo_tf_maior.

# Parte 5 — Regressão com California Housing

## 28. Carregando o dataset

O **California Housing** normalmente é obtido pelo scikit-learn na primeira execução.

Neste laboratório utilizamos a cópia local:

```text
dados/california_housing.csv
```

Assim o exercício de regressão funciona mesmo sem acesso à internet.

In [ ]:
if not CALIFORNIA_CSV.exists():
    raise FileNotFoundError(
        "Arquivo 'dados/california_housing.csv' não encontrado. "
        "Prepare o dataset antes da aula usando a célula opcional "
        "de preparação ou distribua a pasta 'dados/' junto com o notebook."
    )

df_housing = pd.read_csv(
    CALIFORNIA_CSV
)

if "MedHouseVal" not in df_housing.columns:
    raise ValueError(
        "A coluna alvo 'MedHouseVal' não foi encontrada no arquivo."
    )

print(
    "Dataset carregado localmente:",
    CALIFORNIA_CSV.resolve()
)

print(
    "Dimensões:",
    df_housing.shape
)

display(
    df_housing.head()
)

## 29. Separando treino e teste

In [ ]:
X_housing = df_housing.drop(columns='MedHouseVal')
y_housing = df_housing['MedHouseVal']
Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_housing, y_housing, test_size=0.20, random_state=RANDOM_STATE
)

## 30. Baseline de regressão linear

In [ ]:
regressao = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', LinearRegression()),
])
regressao.fit(Xh_train, yh_train)
pred_housing = regressao.predict(Xh_test)
mae = mean_absolute_error(yh_test, pred_housing)
rmse = mean_squared_error(yh_test, pred_housing) ** 0.5
r2 = r2_score(yh_test, pred_housing)
print(f'MAE : {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R²  : {r2:.4f}')

## DESAFIO 8 — Interprete as métricas

Explique MAE, RMSE e R².

In [ ]:
# TODO: escreva sua interpretação.

# Parte 6 — Data Leakage

## 31. Exemplo INCORRETO

In [ ]:
# NÃO FAÇA ISSO EM UM PROJETO REAL.
scaler_errado = StandardScaler()
X_tudo_escalado = scaler_errado.fit_transform(X_cancer)
X_train_errado, X_test_errado, y_train_errado, y_test_errado = train_test_split(
    X_tudo_escalado, y_cancer, test_size=0.20, random_state=RANDOM_STATE, stratify=y_cancer
)

O scaler foi ajustado usando também registros que deveriam estar reservados para teste.

## 32. Forma correta

In [ ]:
modelo_sem_leakage = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', LogisticRegression(max_iter=1000)),
])
modelo_sem_leakage.fit(Xc_train, yc_train)

## DESAFIO 9 — Encontre o leakage

```python
df['media_global'] = df['valor'].mean()
X_train, X_test = train_test_split(df)
```

Há risco de leakage? Como reorganizar?

In [ ]:
# TODO: responda aqui.

# Parte 7 — Comparação final

In [ ]:
comparacao_cancer = pd.DataFrame({
    'modelo': ['Logistic Regression', 'MLP scikit-learn', 'TensorFlow/Keras'],
    'accuracy': [
        accuracy_score(yc_test, pred_log),
        accuracy_score(yc_test, pred_mlp),
        accuracy_score(yc_test, pred_tf),
    ],
    'precision': [
        precision_score(yc_test, pred_log),
        precision_score(yc_test, pred_mlp),
        precision_score(yc_test, pred_tf),
    ],
    'recall': [
        recall_score(yc_test, pred_log),
        recall_score(yc_test, pred_mlp),
        recall_score(yc_test, pred_tf),
    ],
})
display(comparacao_cancer)

## DESAFIO 10 — Qual você colocaria em produção?

Considere desempenho, complexidade, manutenção, explicabilidade e volume de dados.

In [ ]:
# TODO: escreva sua decisão.

# Atividade Prática — Projeto Final

Escolha um dataset público (Kaggle, UCI, OpenML, dados.gov.br ou equivalente).

## Requisitos

1. Problema e pergunta de análise
2. Origem e descrição do dataset
3. Etapas do KDD
4. Baseline
5. Segundo modelo
6. Métricas adequadas
7. Pelo menos dois gráficos
8. Conclusão, limitações e risco de leakage

## Espaço para o projeto

In [ ]:
# Dataset escolhido:
# Fonte:
# Problema:

In [ ]:
# Carregamento e inspeção

In [ ]:
# Preparação / KDD

In [ ]:
# Baseline

In [ ]:
# Segundo modelo

In [ ]:
# Avaliação

In [ ]:
# Visualizações

In [ ]:
# Conclusões

# Encerramento

Machine Learning não começa no algoritmo. Começa na compreensão do problema, na qualidade dos dados e na forma correta de avaliar o resultado.